In [1]:
import pandas as pd
import numpy as np
import sys
parent_path = '/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline'
sys.path.append(parent_path)
from SynOmics.metrics.fidelity.PairwiseSimilarity import PairwiseSimilarity
from SynOmics.processing.metadata import MetaData
from SynOmics.processing.postprocessing import post_masking
from typing import Dict, Optional, Tuple
import seaborn as sns
import matplotlib.pyplot as plt
import itertools
from scipy import stats
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib as mpl
from numpy.random import default_rng
from BroadUtility import plot_violin

###### Data. ######
or_data = pd.read_csv("../Data/original_data.csv", index_col = 0)
transcriptomics_or_data = or_data.iloc[:,58:]
metadata = MetaData.get_metadata(data = transcriptomics_or_data, 
                                 threshold_unique_values = 10, 
                                    ordinal_features = None)


seeds = [42]

results_all = {}
n = int(1e6)
rng = default_rng(42) 
for seed in seeds:
    print(f"---Seed {seed}---")
    syn_datas = [f"avatarsk10_{seed}", f"synthpop_{seed}"]
    results_seed = {}
    visulize_results_dict = {}
    visulize_results_dict_filtered = {}
    for syn_data in syn_datas:
        print(f"---Method {syn_data}---")
        syn_df = pd.read_csv(f"../Data/{syn_data}.csv", index_col = 0)
        transcriptomics_syn_df = syn_df.iloc[:,58:]
        # masked_syn_clinical_df = post_masking(clinical_syn_df)
        pw_computer = PairwiseSimilarity(
        original_data = transcriptomics_or_data,
        synthetic_data = transcriptomics_syn_df,
        metadata = metadata,
        output_dir = f"PairwiseTranscriptomicsSimi_Demo/PairwiseTranscriptomicsSimi_Demo{seed}",
        verbose = True,
        save = True,
        name = syn_data
    )
        result = pw_computer.get_pairwise_scores(method = "spearman")
        results_seed[syn_data] =result
        correlations_scores = result["PairwiseScore"]
        or_correlations = result["OriginalCorrelation"]
        indices = rng.choice(correlations_scores.shape[0], size=n, replace=False)
        sampling_scores = correlations_scores[indices]
        sampling_or_correlations = or_correlations[indices]
        filtered_indices =  np.where(np.absolute(sampling_or_correlations) > 0.5)[0]
        chosing_scores = sampling_scores[filtered_indices]
        visulize_results_dict[syn_data] = sampling_scores
        visulize_results_dict_filtered[syn_data] = chosing_scores
    title_sampling = f'Distribution of pairwise similarity scores (Transcriptomics Features)'
    # fig_sampling, ax = plot_violin(
    #     visulize_results_dict,
    #     title=title_sampling,
    #     annotate=True,
    #     figsize=(10, 5),
    # )
                                    
    # fig_sampling.savefig(f"PairwiseTranscriptomicsSimi_Demo/PairwiseTranscriptomicsSimi_Demo{seed}/Benchmark_PaiwiseTranscriptomics_{seed}.png", dpi=300, bbox_inches='tight')
    
    title_filtered = f'Distribution of pairwise similarity scores (Transcriptomics Features)'
    # fig_filtered, ax = plot_violin(
    #     visulize_results_dict_filtered,
    #     title=title_filtered,
    #     annotate=True,
    #     figsize=(10, 5),
    # )
    # fig_filtered.savefig(f"PairwiseTranscriptomicsSimi_Demo/PairwiseTranscriptomicsSimi_Demo{seed}/Benchmark_PaiwiseTranscriptomics_Filter_{seed}.png", dpi=300, bbox_inches='tight')


    

---Seed 42---
---Method avatarsk10_42---
--- System & Process Info ---
Current Date and Time (UTC): 2025-12-10 22:30:43
Current User's Login: trinhtc
CPU Model: x86_64
Physical Cores: 64
Logical Processors: 64
Process RAM before execution: 932.37 MB

--- GPU Info ---
GPU monitoring failed. Ensure 'nvidia-ml-py' is installed and NVIDIA drivers are accessible.
Error: module 'nvidia_smi' has no attribute 'nvidia_smi_lib'

--- Function Execution ---
Processing data
Calculating correlation matrix for both original data and synthetic data. This matrix is mixed between spearman correlation and Cramér's V.

Summary of Pairwise score:
Summary of Pairwise Score Matrix:
  Number of elements: 837775711
  Min value: 0.5533
  Max value: 1.0000
  Mean: 0.9551
  Median: 0.9676
  Standard deviation: 0.0449

--- Resource Usage Summary ---
Execution time: 292.642007 seconds
Process RAM after execution: 20241.79 MB
Process RAM used by function: 19309.42 MB
Average per-core CPU during execution: [11.0, 0.6

In [9]:
import numpy as np
result_synth = results_seed['synthpop_42']
# 1. Create a boolean mask using the NumPy absolute function
mask = np.abs(result_synth['OriginalCorrelation']) >= 0.5

# 2. Use the mask to filter the correlation values
high_corr = result_synth['OriginalCorrelation'][mask]

# 3. Use the same mask to filter the pairwise scores
high_corr_scores = result_synth['PairwiseScore'][mask]

print(f"Number of high correlation pairs found: {len(high_corr_scores)}")
print('Mean', np.mean(high_corr_scores))

Number of high correlation pairs found: 46595106
Mean 0.9569086392979094


In [10]:
import numpy as np
result_ava = results_seed['avatarsk10_42']
# 1. Create a boolean mask using the NumPy absolute function
mask = np.abs(result_ava['OriginalCorrelation']) >= 0.5

# 2. Use the mask to filter the correlation values
high_corr = result_ava['OriginalCorrelation'][mask]

# 3. Use the same mask to filter the pairwise scores
high_corr_scores = result_ava['PairwiseScore'][mask]

print(f"Number of high correlation pairs found: {len(high_corr_scores)}")
print('Mean', np.mean(high_corr_scores))

Number of high correlation pairs found: 46595106
Mean 0.8449937744075402


In [5]:
import numpy as np
result_synth = results_seed['synthpop_42']
# 1. Create a boolean mask using the NumPy absolute function
mask = np.abs(result_synth['OriginalCorrelation']) <= 0.5

# 2. Use the mask to filter the correlation values
high_corr = result_synth['OriginalCorrelation'][mask]

# 3. Use the same mask to filter the pairwise scores
high_corr_scores = result_synth['PairwiseScore'][mask]

print(f"Number of high correlation pairs found: {len(high_corr_scores)}")
print('Mean', np.mean(high_corr_scores))

Number of high correlation pairs found: 791180605
Mean 0.9447488913425581


In [6]:
import numpy as np
result_ava = results_seed['avatarsk10_42']
# 1. Create a boolean mask using the NumPy absolute function
mask = np.abs(result_ava['OriginalCorrelation']) <= 0.5

# 2. Use the mask to filter the correlation values
high_corr = result_ava['OriginalCorrelation'][mask]

# 3. Use the same mask to filter the pairwise scores
high_corr_scores = result_ava['PairwiseScore'][mask]

print(f"Number of high correlation pairs found: {len(high_corr_scores)}")
print('Mean', np.mean(high_corr_scores))

Number of high correlation pairs found: 791180605
Mean 0.961611225668096
